In [2]:
from typing import TypedDict, List, Literal, Annotated
# from dotenv.variables import Literal
from langgraph.graph import StateGraph, START, END
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_text_splitters import RecursiveCharacterTextSplitter
from docx_markdown_loader import load_docx_as_markdown
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import gradio as gr
import os
from openai import OpenAI

In [3]:
load_dotenv(verbose=True)

True

In [4]:
openai_api_key = os.getenv("OPENAI_API_KEY")
print(f"OpenAI API KEY:{openai_api_key[:10]} loaded")

OpenAI API KEY:sk-proj-uC loaded


In [5]:
opeanai_client = OpenAI(api_key=openai_api_key)
print("OpenAi Client initialised")

OpenAi Client initialised


In [6]:
FILEPATH = "TCS_Network_KB_SOPs.docx"
print(f"Filepath:{FILEPATH}")

Filepath:TCS_Network_KB_SOPs.docx


In [7]:
# convert the docx into Markdown formatted text
raw_doc = load_docx_as_markdown(FILEPATH)
print(f"{len(raw_doc)} documents loaded and total number of character is {len(raw_doc[0].page_content)}")
raw_doc


1 documents loaded and total number of character is 19245


[Document(metadata={'source': 'TCS_Network_KB_SOPs.docx'}, page_content="NETWORK INFRASTRUCTURE\n\nKnowledge Base — Standard Operating Procedures\n\nTata Consultancy Services\n\nNetwork Infrastructure & Security Division\n\n| Document ID | Version | Date | Classification |\n| --- | --- | --- | --- |\n| TCS-NET-KB-002 | v1.0 | June 2025 | Internal Confidential |\n\n| Sections Covered |\n| --- |\n| 1. F5 BIG-IP Firmware Upgrade |\n| 2. F5 SSL Certificate Renewal |\n| 3. Cisco Nexus Switch Firmware Upgrade |\n| 4. F5 BIG-IP Failover Testing |\n| 5. Switch VLAN Configuration |\n| 6. Switch Trunk Configuration |\n| 7. BGP Troubleshooting |\n\n# 1. F5 BIG-IP Firmware Upgrade Process\n\nThis SOP covers the end-to-end process for upgrading F5 BIG-IP software in a high-availability (HA) Active/Standby pair. Always upgrade the Standby unit first.\n\n## 1.1 Pre-Upgrade Checklist\n\n| Check Item | Command / Action | Expected Result |\n| --- | --- | --- |\n| Verify HA Pair Status | show sys failove

In [8]:
# Split the SOP document into chunks with some overlap
doc_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=["\n# ", "\n## ", "\n### ", "\n\n", "\n", " ", ""],
)
document = doc_splitter.split_documents(raw_doc)
document

[Document(metadata={'source': 'TCS_Network_KB_SOPs.docx'}, page_content='NETWORK INFRASTRUCTURE\n\nKnowledge Base — Standard Operating Procedures\n\nTata Consultancy Services\n\nNetwork Infrastructure & Security Division\n\n| Document ID | Version | Date | Classification |\n| --- | --- | --- | --- |\n| TCS-NET-KB-002 | v1.0 | June 2025 | Internal Confidential |\n\n| Sections Covered |\n| --- |\n| 1. F5 BIG-IP Firmware Upgrade |\n| 2. F5 SSL Certificate Renewal |\n| 3. Cisco Nexus Switch Firmware Upgrade |\n| 4. F5 BIG-IP Failover Testing |\n| 5. Switch VLAN Configuration |\n| 6. Switch Trunk Configuration |\n| 7. BGP Troubleshooting |'),
 Document(metadata={'source': 'TCS_Network_KB_SOPs.docx'}, page_content="# 1. F5 BIG-IP Firmware Upgrade Process\n\nThis SOP covers the end-to-end process for upgrading F5 BIG-IP software in a high-availability (HA) Active/Standby pair. Always upgrade the Standby unit first.\n\n## 1.1 Pre-Upgrade Checklist\n\n| Check Item | Command / Action | Expected 

In [9]:
print(f"Raw documnet splitted in {len(document)} chunks")
print(f"Total number of character in chunk is {len(document[0].page_content)}")
print(f"metadata : {document[0].metadata}")
print(f"Content : {document[0].page_content}")

Raw documnet splitted in 22 chunks
Total number of character in chunk is 549
metadata : {'source': 'TCS_Network_KB_SOPs.docx'}
Content : NETWORK INFRASTRUCTURE

Knowledge Base — Standard Operating Procedures

Tata Consultancy Services

Network Infrastructure & Security Division

| Document ID | Version | Date | Classification |
| --- | --- | --- | --- |
| TCS-NET-KB-002 | v1.0 | June 2025 | Internal Confidential |

| Sections Covered |
| --- |
| 1. F5 BIG-IP Firmware Upgrade |
| 2. F5 SSL Certificate Renewal |
| 3. Cisco Nexus Switch Firmware Upgrade |
| 4. F5 BIG-IP Failover Testing |
| 5. Switch VLAN Configuration |
| 6. Switch Trunk Configuration |
| 7. BGP Troubleshooting |


In [10]:
# Let's initialize our embeddings model. Note that we will use OpenAI's embedding model
db_name = "vector_db_langgraph_v1"
openai_embeddings = OpenAIEmbeddings(api_key=openai_api_key)
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=openai_embeddings).delete_collection()
vector_store = Chroma.from_documents(
    documents=document,
    embedding= openai_embeddings,
    persist_directory=db_name,
)
retriever = vector_store.as_retriever(search_kwargs={"k":4})
print(f"Vector store initialised with {vector_store._collection.count()} vectors")


Vector store initialised with 22 vectors


In [11]:
MODEL = "gpt-4o-mini"
llm = ChatOpenAI(
    model=MODEL,
    temperature=0.3,
    api_key=openai_api_key,
    max_tokens=2000,
)

In [12]:
'''
Define State. Think of state as a **shared clipboard** that gets passed from node to node. Every node
receives the *entire* clipboard as input, and hands back only the fields it filled in —
LangGraph then updates the master clipboard with those changes before passing it to the
next node.
'''
class RAGState(TypedDict):
    question : str
    context : str
    answer : str
    grade : str
    retry_count : int

In [13]:
'''
Define node. A node is just a regular Python function with one rule: it takes the state dict as its
only argument, and returns a dict of the fields it wants to update. That's the entire contract

it take `state["question"]`, search the vector store, and turn the results into one
formatted context string. Notice it doesn't touch `answer` at all — it only returns the
one key it's responsible for, `context`. LangGraph merges that into the state without
disturbing anything else.
'''
def retrieve(state: RAGState) -> dict:
    docs : List[Document] = retriever.invoke(state["question"])
    context_part = []
    for doc in docs:
        source = doc.metadata.get("source", "unknown")
        context_part.append(f"Source : {source} \n Contents {doc.page_content}")
    context = "\n\n-----\n\n".join(context_part)

    return {"context": context}

In [14]:
class GradeContext(BaseModel):
    binary_score : Literal["sufficient", "insufficient"] = Field(
        description=("sufficient' if the retrieved context contains enough information to"
                     "directly answer the question. 'insufficient' if the context is off-topic "
                    "or too thin to answer from.")
    )

In [15]:
grade_llm = ChatOpenAI(
    model=MODEL,
    temperature=0,
    api_key=openai_api_key,
).with_structured_output(GradeContext)

In [16]:
def grade_document(state: RAGState) -> dict:
    grade_prompt = (
        f"Questions : {state['question']} \n\n"
        f"Retrieved context : {state['context']} \n\n"
        "does this context contain enough information to directly answer the question?"
    )
    result : GradeContext = grade_llm.invoke([HumanMessage(content=grade_prompt)])
    return {"grade": result.binary_score}

In [17]:
rewriter_llm = ChatOpenAI(model=MODEL, temperature=0.3, api_key=openai_api_key)

In [18]:
def transform_query(state:RAGState)-> dict:
     rewrite_prompt = (
        "You rewrite vague network-engineering questions into precise queries for a "
        "vector search over SOP documents covering F5 BIG-IP, Cisco Nexus, VLAN/trunk "
        "configuration, and BGP troubleshooting. Rewrite the question below to be more "
        "specific and retrieval-friendly. Return ONLY the rewritten question.\n\n"
        f"Original question: {state['question']}"
    )
     rewritten = rewriter_llm.invoke([HumanMessage(content=rewrite_prompt)])
     return {"question":rewritten.content.strip(), "retry_count": state["retry_count"]+1}

In [19]:
'''
node 2: `generate`
Job: take `state["question"]` and the `state["context"]` that `retrieve` just filled in,
build the same system prompt discipline (answer only from
context, reproduce commands exactly, cite the section), call the LLM, and return the
answer.
'''
SYSTEM_PROMPT_TEMPLATE = f"""
You are a Network Operations Knowledge Assistant for TCS's Network Infrastructure & Security Division.

Answer using ONLY the SOP content in the context below.

Guidelines:
1. If the context doesn't contain the answer, say "I don't have that information in the knowledge base" — do not guess.
2. Reproduce CLI/tmsh commands exactly as they appear in the context.
3. Preserve ordered steps in the order given.
4. Include Pass Criteria / Expected Result tables when present, so the engineer can confirm success.
5. Surface rollback or emergency steps proactively for failure-related questions.
6. Cite which section the answer came from (e.g., "Per Section 1.5 — Rollback Procedure").
7. Keep answers concise and operational.
8. Context quality flag for this turn: {grade}. If "insufficient", be upfront that the knowledge base didn't have a strong match rather than stretching the context to fit.

Context:
{context}
"""

In [20]:
def generate(state: RAGState) -> dict:
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=state["context"], grade=state["grade"])
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=state["question"])])
    return {"answer" : response.content}

In [21]:
MAX_RETRIES = 2

In [22]:
def decide_next_step(state: RAGState) -> Literal["generate", "transform_query"]:
    if state["grade"] == "sufficient" or state["retry_count"] >= MAX_RETRIES:
        return "generate"
    return "transform_query"

In [23]:
'''
what is an "edge", and what does `compile()` do?

An edge is just an arrow: "after this node finishes, run that node next." We declare
the whole map with `add_edge(from_node, to_node)`. `START` and `END` are special
built-in markers, not real nodes — they just say where the graph begins and where it's
done.

`add_node("retrieve", retrieve)` registers the function under a *name* — that name is
what edges refer to, and it's also what shows up later when we inspect or visualize the
graph. The name doesn't have to match the function name, but keeping them the same avoids
confusion.

`.compile()` takes the node/edge definitions and turns them into something runnable
(`app`, below). Nothing has executed yet at this point — compiling just validates the
graph shape and prepares it to be invoked.
'''
graph_builder = StateGraph(RAGState)
graph_builder.add_node("retrieve", retrieve)
graph_builder.add_node("grade_document", grade_document)
graph_builder.add_node("transform_query", transform_query)
graph_builder.add_node("generate", generate)

graph_builder.add_edge(START, "retrieve")
graph_builder.add_edge("retrieve", "grade_document")
graph_builder.add_conditional_edges("grade_document", decide_next_step, {"generate": "generate", "transform_query": "transform_query"})
graph_builder.add_edge("transform_query", "retrieve")
graph_builder.add_edge("generate", END)

app = graph_builder.compile()
print(f"graph compiled node {list(app.get_graph().nodes)}")

graph compiled node ['__start__', 'retrieve', 'grade_document', 'transform_query', 'generate', '__end__']


In [24]:
result = app.invoke({"question": "What is the rollback procedure if the F5 upgrade fails?", "retry_count": 0})
print("Retries used:", result["retry_count"])
print("Grade:", result["grade"])
print(result["answer"])

Retries used: 0
Grade: sufficient
Per Section 1.5 — Rollback Procedure:

If the upgrade fails or any virtual server is down after the upgrade, execute the rollback immediately.

1. Boot back to the previous volume: 
   ```
   tmsh reboot volume HD1.1
   ```

2. Restore UCS backup if the configuration is corrupt: 
   ```
   tmsh load sys ucs /var/local/ucs/pre_upgrade.ucs
   ```

3. Verify all services are restored.

4. Raise an incident ticket and contact F5 TAC: 1-888-882-7535.


In [25]:
result = app.invoke({"question": "nexus is down", "retry_count": 0})
print("Retries used:", result["retry_count"])
print("Grade:", result["grade"])
print(result["answer"])

Retries used: 2
Grade: insufficient
I don't have that information in the knowledge base.


In [26]:
def question_answer(query, history):
    result = app.invoke({"question" :query, "retry_count": 0})
    return result["answer"]

In [27]:
demo = gr.ChatInterface(
    fn=question_answer,
    title="NetSOP-RAG (LangGraph —  v1)",
    description="Ask about F5 BIG-IP, Cisco Nexus, VLAN/trunk, or BGP SOPs."
)
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://adbd1bc8f1b99a5d85.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
